<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/LangStudio/RSI_Max_Agent_Checker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import yfinance as yf
import numpy as np

# --- Configuration ---
RSI_LENGTH = 7
RSI_THRESHOLD = 80  # Changed from 25 to 70 for overbought detection
CSV_FILE = "OptionVolume.csv"

def calculate_rsi_wilder(series, period):
    """Matches Yahoo Finance Precision (Wilder's Smoothing)"""
    delta = series.diff()
    gain = (delta.where(delta > 0, 0))
    loss = (-delta.where(delta < 0, 0))

    # Use EWM with alpha = 1/period (Wilder's)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def main():
    print(f"--- RSI-{RSI_LENGTH} Overbought Audit Tool ---")

    # 1. Load Symbols
    try:
        df_csv = pd.read_csv(CSV_FILE)
        symbol_col = [c for c in df_csv.columns if 'symbol' in c.lower()][0]
        symbols = df_csv[symbol_col].str.strip().unique().tolist()
        print(f"Found {len(symbols)} symbols in {CSV_FILE}")
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return

    matches = []

    # 2. Fetch and Calculate
    print(f"Scanning for RSI > {RSI_THRESHOLD}...") # Updated log message
    for s in symbols:
        try:
            # Fetch enough data for the RSI warm-up
            df = yf.download(s, period="200d", interval="1d", progress=False)
            if df.empty: continue

            # Ensure we handle multi-index columns if they exist
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            # Calculate RSI
            df['RSI'] = calculate_rsi_wilder(df['Close'], period=RSI_LENGTH)

            # Extract current values
            current_rsi = df['RSI'].iloc[-1]
            current_price = df['Close'].iloc[-1]

            # Logic Flip: Check if RSI is GREATER than threshold
            if not np.isnan(current_rsi) and current_rsi > RSI_THRESHOLD:
                matches.append({
                    "Symbol": s,
                    "Price": round(float(current_price), 2),
                    "RSI": round(float(current_rsi), 2)
                })
                print(f"✅ MATCH (Overbought): {s} (RSI: {round(current_rsi, 2)})")

        except Exception as e:
            # print(f"Skipping {s}: {e}")
            continue

    # 3. Final Summary
    print("\n" + "="*30)
    if matches:
        results_df = pd.DataFrame(matches)
        print("VERIFICATION RESULTS (OVERBOUGHT):")
        print(results_df.to_string(index=False))
        results_df.to_csv("audit_results.csv", index=False)
        print(f"\nSaved {len(matches)} matches to audit_results.csv")
    else:
        print("No stocks currently meet the criteria.")
    print("="*30)

if __name__ == "__main__":
    main()

--- RSI-7 Overbought Audit Tool ---
Found 150 symbols in OptionVolume.csv
Scanning for RSI > 80...


/tmp/ipykernel_395/3012101016.py:43: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(s, period="200d", interval="1d", progress=False)
/tmp/ipykernel_395/3012101016.py:43: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(s, period="200d", interval="1d", progress=False)
/tmp/ipykernel_395/3012101016.py:43: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(s, period="200d", interval="1d", progress=False)
/tmp/ipykernel_395/3012101016.py:43: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(s, period="200d", interval="1d", progress=False)
/tmp/ipykernel_395/3012101016.py:43: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(s, period="200d", interval="1d", progress=False)
/tmp/ipykernel_395/3012101016.py:43: FutureWarning: YF.download() has changed argumen

✅ MATCH (Overbought): USO (RSI: 88.85)


/tmp/ipykernel_395/3012101016.py:43: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(s, period="200d", interval="1d", progress=False)
/tmp/ipykernel_395/3012101016.py:43: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(s, period="200d", interval="1d", progress=False)
/tmp/ipykernel_395/3012101016.py:43: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(s, period="200d", interval="1d", progress=False)
/tmp/ipykernel_395/3012101016.py:43: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(s, period="200d", interval="1d", progress=False)
/tmp/ipykernel_395/3012101016.py:43: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(s, period="200d", interval="1d", progress=False)
/tmp/ipykernel_395/3012101016.py:43: FutureWarning: YF.download() has changed argumen


VERIFICATION RESULTS (OVERBOUGHT):
Symbol  Price   RSI
   USO 118.39 88.85

Saved 1 matches to audit_results.csv


/tmp/ipykernel_395/3012101016.py:43: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(s, period="200d", interval="1d", progress=False)
